In [1]:
import sys
import os
current_path = os.path.abspath('')
data_path = '/'.join(current_path.split('/')[:-1]) + '/data/'
script_path = '/'.join(current_path.split('/')[:-1]) + '/scripts/'
sys.path.append(script_path)
sys.path.append('/home/fkunneman/.local/lib/python3.10/site-packages/')
sys.path.append('/usr/lib/python3/dist-packages/')

In [2]:
import torch
import gc
import importlib
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_community.vectorstores import Chroma
import chromadb
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings

import instruction_agent

/home/fkunneman/.local/lib/python3.10/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [3]:
### Load models for chatbot and rag in memory
with open('/home/fkunneman/hf.txt') as fin:
    hf = fin.read().strip()
os.environ["HF_TOKEN"] = hf
torch.cuda.empty_cache()
gc.collect()
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained('BramVanroy/GEITje-7B-ultra', use_fast=True)
tokenizer.pad_token = tokenizer.eos_token # Ensure padding token is set
# Initialize language model
chatmodel = AutoModelForCausalLM.from_pretrained('BramVanroy/GEITje-7B-ultra', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")
#chatmodel = AutoModelForCausalLM.from_pretrained('robinsmits/Qwen1.5-7B-Dutch-Chat', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")     
# set pipeline
pipe = pipeline(
    "text-generation",
    model=chatmodel,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=250, # Limit the number of generated tokens for concise responses
    do_sample=True, # Enable sampling for more varied responses
    temperature=0.3 # Control creativity
)
llm = HuggingFacePipeline(pipeline=pipe)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [14]:
chroma_client = chromadb.Client()
#embedding_model = SentenceTransformer('jegormeister/bert-base-dutch-cased')
collection = chroma_client.create_collection(
    name="instruction_db2",
    configuration={
        "hnsw": {
            "space": "cosine",
            "ef_construction": 30
        }
    }
)

In [5]:
instructions_travel = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_ov_stripped.csv'
instructions_passport = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_paspoort_stripped_v2.csv'
pat = data_path + 'opslag_inclusieve_spraakassistent_project/patterns_v2.csv'
qa_travel = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_ov_v5.csv'
qa_passport = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_paspoort.csv'
nav = data_path + 'opslag_inclusieve_spraakassistent_project/navigation.csv'


In [6]:
importlib.reload(instruction_agent)
assistant = instruction_agent.InstructAgent(llm,'travel')
assistant.prepare_patterns(pat)
assistant.prepare_instructions(instructions_travel)
assistant.setup_rag(collection,[['qa',qa_travel],['nav',nav]])

In [16]:
importlib.reload(instruction_agent)
assistant = instruction_agent.InstructAgent(llm,'passport')
assistant.prepare_instructions(instructions_passport)
assistant.prepare_patterns(pat)
assistant.setup_rag(collection,[['qa',qa_passport],['nav',nav]])

INP Nee OUTP Dan wachten we nog even. Neem je tijd!
INP Nog niet OUTP Dan wachten we nog even. Neem je tijd!
INP Waar is de blauwe tekst? OUTP De blauwe tekst 'Paspoort, ID-kaart en rijbewijs' staat links onder de blauwe tekst 'verhuizing doorgeven'
INP Waar moet ik klikken? OUTP De blauwe tekst 'Paspoort, ID-kaart en rijbewijs' staat links onder de blauwe tekst 'verhuizing doorgeven'
INP Waar? OUTP De blauwe tekst 'Paspoort, ID-kaart en rijbewijs' staat links onder de blauwe tekst 'verhuizing doorgeven'
INP Waar is de blauwe tekst? OUTP De blauwe tekst 'Paspoort 18 jaar en ouder' staat onder de zwarte tekst 'Paspoort aanvragen'
INP Waar moet ik klikken? OUTP De blauwe tekst 'Paspoort 18 jaar en ouder' staat onder de zwarte tekst 'Paspoort aanvragen'
INP Waar? OUTP De blauwe tekst 'Paspoort 18 jaar en ouder' staat onder de zwarte tekst 'Paspoort aanvragen'
INP Waar is de blauwe tekst? OUTP De blauwe tekst 'afspraak maken' staat onder de zwarte tekst 'Aanvragen'
INP Waar moet ik klikken

In [17]:
assistant.chat()

Hallo! Ik ga je helpen met het maken van een afspraak voor een paspoort. Je kan altijd een vraag stellen. Ben je er klaar voor?


You:  ja


{'ids': [['0f03eea1-c19e-469f-8d85-28e49af59e75', '517c89dd-bb1e-4db3-a98b-de648c9ced37', 'c7b07538-0065-4de0-ae33-b1824f921372']], 'embeddings': None, 'documents': [['Ja', 'Ja', 'Wat zeggen jij?']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'action': 'Confirm', 'step_context': 'all', 'type': 'nav'}, {'type': 'nav', 'action': 'Confirm', 'step_context': 'all'}, {'step_context': 'all', 'type': 'nav', 'action': 'current step'}]], 'distances': [[-4.76837158203125e-07, -4.76837158203125e-07, 0.4311416745185852]]} b 50
Agent: Super! Stap 1. Onder 'persoon/personen' kun je kiezen voor hoeveel personen je het paspoort aanvraagt. Druk op de plus knop om mensen toe te voegen aan het witte vakje.


You:  volgende


{'ids': [['0060ad57-82f9-46ff-9f2a-0f5cb4cf7abb', '1f00b466-9902-4fe3-aaea-620b1c94d20d', '239eaed6-c0d0-4af4-980f-35cad44a3be2']], 'embeddings': None, 'documents': [['Volgende', 'Volgende', 'Vorige']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'action': 'next step', 'type': 'nav', 'step_context': 'all'}, {'action': 'next step', 'type': 'nav', 'step_context': 'all'}, {'action': 'previous step', 'type': 'nav', 'step_context': 'all'}]], 'distances': [[-3.5762786865234375e-07, -3.5762786865234375e-07, 0.4170788526535034]]} 1 88
Agent: Klik op de knop '2 Kies een locatie'


...

{'ids': [['69bc03ad-276e-44b5-bd87-089fa6a71755', '22bf790c-efc6-4d47-8d97-7e360d1efc8c', 'a8fc7eea-8d8c-4116-af71-4b485ddf68ea']], 'embeddings': None, 'documents': [['dankjewel', 'dankjewel', 'dankje']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'step_context': '', 'type': 'instruct', 'answer': ' Graag gedaan!'}, {'answer': ' Graag gedaan!', 'type': 'instruct', 'step_context': ''}, {'answer': ' Graag gedaan!', 'type': 'instruct', 'step_context': ''}]], 'distances': [[0.19646579027175903, 0.19646579027175903, 0.2739713191986084]]} 11 72
Agent:  Graag gedaan!


You:  volgende


{'ids': [['0060ad57-82f9-46ff-9f2a-0f5cb4cf7abb', '1f00b466-9902-4fe3-aaea-620b1c94d20d', '79ffdefc-a4b2-46bd-bd66-05ac413842b4']], 'embeddings': None, 'documents': [['Volgende', 'Volgende', 'Vorige']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'step_context': 'all', 'type': 'nav', 'action': 'next step'}, {'action': 'next step', 'type': 'nav', 'step_context': 'all'}, {'step_context': 'all', 'type': 'nav', 'action': 'previous step'}]], 'distances': [[-3.5762786865234375e-07, -3.5762786865234375e-07, 0.4170788526535034]]} 11 88
Agent: Je moet meenemen: je oude paspoort, geld of een bankpas om voor het paspoort te betalen en een foto van jezelf in kleur. Deze foto moet voldoen aan de eisen voor pasfoto's. En daarmee heb je de taak volbracht! Heb je nog vragen? Stel ze gerust. Geen vragen? Zeg dan 'klaar'


You:  klaar


{'ids': [['b9e12aed-6f0c-4574-a0e1-c8b8700dd24e', '4254872a-3110-447b-ac52-3917fbe857d7', 'a8fc7eea-8d8c-4116-af71-4b485ddf68ea']], 'embeddings': None, 'documents': [['Klaar', 'Klaar', 'dankje']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'type': 'nav', 'step_context': 'all', 'action': 'Done'}, {'type': 'nav', 'action': 'Done', 'step_context': 'all'}, {'step_context': '', 'type': 'instruct', 'answer': ' Graag gedaan!'}]], 'distances': [[2.384185791015625e-07, 2.384185791015625e-07, 0.5340769290924072]]} e 80
Agent: Tot ziens!
